# Ingestão Bronze: ADLS Gen2 ➔ Delta Lake (Bronze)
Este notebook realiza:
- Autenticação via Service Principal (credenciais somente via `.env`).
- Polling recursivo de novos arquivos `ecommerce_enderecos.parquet` (micro-lote = 1 arquivo).
- Download do `.parquet` em memória (SDK) e conversão para DataFrame Spark, **sem nenhuma transformação** nos dados.
- Colunas de auditoria: `bronze_ingested_at` (`current_timestamp()`) e `bronze_source_file` (arquivo de origem).
- Escrita Delta (`append`) particionada por data de ingestão (`ano`, `mes`, `dia`, `hora`).

In [ ]:
import os
import io
import time
import pandas as pd
from dotenv import load_dotenv
from azure.identity import ClientSecretCredential
from azure.storage.filedatalake import DataLakeServiceClient
from pyspark.sql import functions as F
from pyspark.sql.functions import current_timestamp, lit, year, month, dayofmonth, hour

# ==============================================================================
# 1. Carregamento Seguro das Credenciais e Parâmetros
# ==============================================================================
load_dotenv(".env")

client_id     = os.getenv("ADLS_CLIENT_ID")
tenant_id     = os.getenv("ADLS_TENANT_ID")
client_secret = os.getenv("ADLS_CLIENT_SECRET")

storage_account_name = "internshipdatalake"
container_name = "raw"
base_directory = "real-time-data"
target_file_name = "ecommerce_enderecos.parquet"
bronze_table = "bronze_ecommerce_enderecos"
poll_interval_seconds = 15

assert client_id and tenant_id and client_secret, "Credenciais do ADLS não encontradas no arquivo .env."

# ==============================================================================
# 2. Conexão com o ADLS via SDK (listagem e download em memória)
# ==============================================================================
credential = ClientSecretCredential(tenant_id=tenant_id, client_id=client_id, client_secret=client_secret)

service_client = DataLakeServiceClient(
    account_url=f"https://{storage_account_name}.dfs.core.windows.net",
    credential=credential
)
file_system_client = service_client.get_file_system_client(file_system=container_name)

print("✔ Conexão com o ADLS estabelecida com sucesso.")

In [ ]:
# ==============================================================================
# 3. Checkpoint: arquivos já presentes na Bronze
# ==============================================================================
control_checkpoint = set()

try:
    if spark.catalog.tableExists(bronze_table):
        control_checkpoint = set(
            r["bronze_source_file"]
            for r in spark.table(bronze_table).select("bronze_source_file").distinct().collect()
        )
        print(f"✔ Histórico carregado: {len(control_checkpoint)} arquivo(s) previamente ingerido(s).")
    else:
        print(f"ℹ Tabela '{bronze_table}' não encontrada. Será criada na primeira ingestão.")
except Exception as e:
    print(f"⚠ Aviso ao verificar checkpoint da tabela: {e}")

In [ ]:
# ==============================================================================
# 4. Loop de Monitoramento: micro-lote Bronze (sem transformação)
# ==============================================================================
def ler_parquet_como_spark(file_path):
    """Baixa o .parquet em memória (SDK) e o converte para DataFrame Spark, sem alterar os dados."""
    file_client = file_system_client.get_file_client(file_path)
    pdf = pd.read_parquet(io.BytesIO(file_client.download_file().readall()))

    # Colunas 100% nulas não têm tipo inferível pelo Spark: são mantidas como string nula
    colunas_nulas = [c for c in pdf.columns if pdf[c].isna().all()]
    df = spark.createDataFrame(pdf.drop(columns=colunas_nulas))
    for c in colunas_nulas:
        df = df.withColumn(c, F.lit(None).cast("string"))
    return df.select(*pdf.columns)


print(f"Iniciando monitoramento de '{target_file_name}' em '{base_directory}'...\n")

while True:
    try:
        paths = file_system_client.get_paths(path=base_directory, recursive=True)

        new_files = [
            item.name for item in paths
            if not item.is_directory and item.name.endswith(target_file_name) and item.name not in control_checkpoint
        ]

        for file_path in new_files:
            print(f"Processando: {file_path}")

            df_raw = ler_parquet_como_spark(file_path)

            # Apenas metadados de auditoria; os dados originais permanecem intactos
            df_bronze = (
                df_raw
                .withColumn("bronze_ingested_at", current_timestamp())
                .withColumn("bronze_source_file", lit(file_path))
                # Partições derivadas da data de ingestão
                .withColumn("ano",  year("bronze_ingested_at"))
                .withColumn("mes",  month("bronze_ingested_at"))
                .withColumn("dia",  dayofmonth("bronze_ingested_at"))
                .withColumn("hora", hour("bronze_ingested_at"))
            )

            (
                df_bronze.write
                .format("delta")
                .mode("append")
                .partitionBy("ano", "mes", "dia", "hora")
                .saveAsTable(bronze_table)
            )

            control_checkpoint.add(file_path)
            print(f"✔ Sucesso: '{file_path}' ingerido na Bronze ({df_bronze.count()} registros).")

        time.sleep(poll_interval_seconds)

    except KeyboardInterrupt:
        print("\nProcesso interrompido manualmente pelo usuário.")
        break
    except Exception as e:
        print(f"\n Erro durante a execução do loop: {e}")
        time.sleep(poll_interval_seconds)